# Irina-RL: a coding agent that improves itself

This notebook runs the `odysseus` coding-agent harness and slowly turns it into a self-improving loop:

- **Acting:** a small local `Qwen2.5-Coder-0.5B` replaces the DeepSeek API as the *student* that actually works in the harness.
- **Teaching:** the DeepSeek provider stays behind as the *teacher* — it judges transcripts (the GRPO reward) and distills goldens (the SFT labels). It never sees hidden tests or expected answers.
- **Learning:** reward-driven updates via GRPO (online), imitation of distilled goldens via SFT (offline).

Every section below explains its **logic**, its **goal**, and what it **needs** before the code. Each code cell is independent of the ones after it except where noted. Run all cells in order.

**Before you run:** give the notebook a GPU runtime (Runtime > Change runtime type > T4) and add a DeepSeek API key in the Colab secret panel under the name `DEEPSEEK_API_KEY`.

**Start here:** run the short  "SYNC + RESTART"  cell right below every time you restart the runtime (or after a push). It pulls the latest code and restarts the kernel cleanly; after reconnecting, choose Runtime > Run all.


In [ ]:
# ⚙️ SYNC + RESTART - run this first if you copied the notebook before a push.
# Pulls the latest code from GitHub into /content/Irina-RL. If anything
# changed, it RESTARTS the kernel so the new session starts clean (stale
# modules from a previous run are the #1 "still running old code" bug).
# After the reconnect, click  Runtime > Run all  again. If nothing changed
# it prints "up to date" and the notebook simply continues.
import os, subprocess

REPO = "/content/Irina-RL"

def _git(*args):
    return subprocess.check_output(["git", "-C", REPO] + list(args),
                                   text=True).strip()

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.check_call(
        ["git", "clone", "https://github.com/rderakhshan/Irina-RL", REPO],
        stdout=subprocess.DEVNULL)
    print("Fresh clone. HEAD:", _git("rev-parse", "HEAD"))
    CHANGED = True
else:
    _git("fetch", "origin")
    head = _git("rev-parse", "HEAD")
    origin = _git("rev-parse", "origin/main")
    CHANGED = head != origin
    if CHANGED:
        _git("reset", "--hard", "origin/main")
        print(f"Updated {head[:8]} -> {_git('rev-parse', 'HEAD')[:8]}.")
    else:
        print("Already up to date:", head[:8])

if CHANGED:
    print("\nCode changed - restarting the kernel now.")
    print("Reconnect, then click  Runtime > Run all  again.")
    os.kill(os.getpid(), 9)
else:
    print("\nContinuing...")


## 1. Install the stack

**Logic.** The harness itself needs almost nothing — it talks to DeepSeek over plain HTTP. What actually needs installing is the *local student*: transformers, peft (LoRA), accelerate, and gradio for the UI. Colab ships torch; `torchao` is uninstalled because a preinstalled version can shadow the torch build and break matrix ops.

**Goal.** A runtime where the acting provider can load Qwen and the training math can run, without constantly re-installing in later cells.

**Need.** T4 (or any) GPU runtime; ~2 minutes.

In [ ]:
# No %%capture here on purpose: pip progress prints live in the log panel,
# so this cell never LOOKS frozen while transformers/peft/gradio download.
!pip install -q --upgrade pip
!pip install -q --progress-bar off "transformers>=4.44" peft accelerate gradio
# Colab sometimes ships a preinstalled torchao that shadows the torch build.
# Remove it only when present (never hangs, never fails on a clean box).
!python -c "import importlib.util,sys; sys.exit(0 if importlib.util.find_spec('torchao') else 1)" && pip uninstall -y -q torchao || echo "torchao not installed - skipping"


## 2. Mount Drive, clone the repo, put it on the path

**Logic.** The insight pool and the repo live on Drive so they survive runtime resets. The repo is cloned next to `src/`, and both the repo root and `src/` are added to `sys.path` — the repo root so `experiments` imports, `src/` so the `back` harness package imports.

**Goal.** A persistent work area and importable packages everywhere in the note-book.

**Need.** A Google account (for Drive). The repo is public, so the plain clone just works — no token needed. The clone is idempotent: re-running after a runtime reset reuses the existing checkout.

In [ ]:
from google.colab import drive, userdata
import os, sys

try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount failed (OK to continue):", e)

REPO_DIR = "/content/Irina-RL"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only --quiet
else:
    !git clone https://github.com/rderakhshan/Irina-RL {REPO_DIR}
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

DRIVE_HOME = "/content/drive/MyDrive/Irina-RL"
POOL_PATH = os.path.join(DRIVE_HOME, "insight_pool.json")
os.makedirs(DRIVE_HOME, exist_ok=True)
print("repo:", REPO_DIR)
print("pool:", POOL_PATH)

## 3. Load the DeepSeek key

**Logic.** The teacher's requests go to the DeepSeek API. The key comes from Colab's secret manager so it never appears in the notebook text; it reaches the provider through `DEEPSEEK_API_KEY` in the environment, which is exactly how `back.provider.api_key()` reads it.

**Goal.** Authenticated teacher calls (judge + golden extraction).

**Need.** A DeepSeek API key registered in the secret panel as `DEEPSEEK_API_KEY`.

In [ ]:
import os
try:
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
    print("key loaded")
except Exception as e:
    print("No DEEPSEEK_API_KEY secret found:", e)

## 4. See the harness run (still DeepSeek)

**Logic.** Before any swap, run the *unmodified* harness once to see the shape of a transcript: a user request, assistant tool calls, tool results, and a final prose answer. Everything we later learn from lives in this neutral `messages` list.

**Goal.** One real transcript to point at in the next sections.

**Need.** The API key from section 3. This cell self-bootstraps (clones the repo and adds `src/` to `sys.path`) so it also works if you jumped here first.

In [ ]:
# Idempotent bootstrap - lets this cell run even if you skipped section 2.
import os, sys
REPO_DIR = "/content/Irina-RL"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only --quiet
else:
    !git clone https://github.com/rderakhshan/Irina-RL {REPO_DIR}
for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Section 4 is a SMOKE TEST: it still runs the real DeepSeek API (the swap
# to Qwen is section 5). With the defaults this takes minutes - 8192 output
# tokens a reply, up to 8 turns, and subagent fan-out. For the demo we trim
# all three: small replies, 4 turns, no subagents. The transcript STILL shows
# the user/tool-call/tool-result/final-prose shape.
import back.provider as provider
provider.MAX_OUTPUT_TOKENS = 1024

from back.harness import Harness

demo_dir = "/content/irina_demo"
os.makedirs(demo_dir, exist_ok=True)
h = Harness(demo_dir, max_turns=4, enable_subagents=False)
final = h.run("Create hello.py that prints the square of 9, then run it.")
print("FINAL:", final)
print("messages:", len(h.messages))
for m in h.messages:
    print(" -", m["role"], ":", str(m.get("text", ""))[:90])


## 5. The swap: teacher keeps DeepSeek, student becomes Qwen

**Logic.** `teacher` binds `back.provider.complete` (DeepSeek) **at import time** so it always judges with the teacher. Only after that capture does `provider_local.install_provider()` overwrite `back.provider.complete` with the local Qwen acting provider. From here on the harness runs the 0.5B student for every rollout — same loop, same tools, cheaper actor.

**Goal.** A harness where acting is cheap/trainable (Qwen) and judging is strong (DeepSeek), with no edits to `src/back`.

**Need.** Imports from section 2 (so both `back` and `experiments` resolve). This cell self-bootstraps the same way as section 4.

In [ ]:
# Idempotent bootstrap - lets this cell run even if you skipped section 2.
import os, sys
REPO_DIR = "/content/Irina-RL"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/rderakhshan/Irina-RL {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1
for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# git pull updates FILES, not the modules already imported into this kernel.
# Drop any stale copies so a re-run of this cell (or a re-run after a pull)
# loads the freshly pulled code instead of an old in-memory module.
for _name in list(sys.modules):
    if (_name == "experiments" or _name.startswith("experiments.")
            or _name == "back" or _name.startswith("back.")):
        del sys.modules[_name]

from experiments.selfimprove import teacher
bound = teacher.rebind()
print("teacher bound to DeepSeek:", bound)
assert bound, ("teacher did not bind to DeepSeek - src/ is not on sys.path "
               "yet; check the bootstrap above. Fix it before grading.")

from experiments.selfimprove import provider_local
provider_local.install_provider()

import back.provider as provider
print("provider.complete is now local:", provider.complete is provider_local.complete)


## 6. Transcript -> chat template

**Logic.** Loads the acting model (first time = first Qwen download), runs one tiny task, and shows the conversion the trainer will actually see: neutral `messages` become OpenAI-style chat dicts with a system prompt and the tool block appended, in exactly the order the harness produced them. `grpo.build_masked` later segments this into per-turn labels.

**Goal.** A concrete `to_chat` result on a real transcript, plus a loaded acting model to reuse in the training sections.

**Need.** Qwen download (~1GB) on first `load`.

In [ ]:
# Idempotent bootstrap - lets this cell run even if you skipped section 2.
import os, sys
REPO_DIR = "/content/Irina-RL"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only --quiet
else:
    !git clone https://github.com/rderakhshan/Irina-RL {REPO_DIR}
for _p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from back.harness import Harness
from experiments.selfimprove import traj, provider_local

provider_local.TEMPERATURE = 0.4
q = Harness("/content/irina_qwen", model=provider_local.DEFAULT_MODEL, max_turns=6, persist=False)
q.run("Append the line 'hello from qwen' to qwen.txt")

system = q.system + traj.tool_block([t.spec for t in q.tools.values()])
chat = traj.to_chat(q.messages, system=system)
print("chat turns:", len(chat))
for m in chat[:6]:
    print(" -", m["role"], ":", m["content"][:70].replace("\n", " "))

# Load the student once — reused by the GRPO and SFT sections.
model, tok, device = provider_local.load(provider_local.DEFAULT_MODEL)
print("device:", device)

## 7. The judge: score a transcript, distill a golden

**Logic.** Two teacher actions: `grade` returns a 0-10 score for the GRPO reward (this is what makes one rollout better than another), and `extract_golden` compresses a good run into a stepwise 'golden reasoning trace' that SFT will imitate. Neither ever sees a hidden test or expected answer — the teacher judges the *journey*.

**Goal.** A numeric reward and a text label for the same transcript, both ready to feed the two learning paths.

**Need.** The API key, and the `q` transcript from section 6.

In [ ]:
score = teacher.grade(q.messages)
golden = teacher.extract_golden(q.messages[0]["text"], q.messages)
print("score (0-10):", score)
print("golden trace:")
print((golden or {}).get("golden", "(none)")[:400])

## 8. Bank the session into the insight pool

**Logic.** When a chat session closes, its golden is a candidate for offline learning. The pool stores these on Drive, accepting only runs the judge scored well. It is append-only until the UI's offline button drains it — collection is automatic, application is a deliberate human decision.

**Goal.** A persistent, Drive-backed bank of SFT labels, visible to the UI.

**Need.** The teacher from section 5 (score + goldens).

In [ ]:
from experiments.selfimprove import insight_pool

pool = insight_pool.InsightPool(POOL_PATH, teacher=teacher)
print(pool.bank(q.messages[0]["text"], q.messages))
print(pool.status())

## 9. Online learning: one GRPO step on a fixed task

**Logic.** The reward-driven path, end to end on one task: roll out a small group of the SAME task at temperature 1.0, judge each transcript, attach LoRA (the acting model stays frozen during rollouts; training needs a trainable head), then take one group-relative-advantage REINFORCE update on the assistant-token log-probs. Returns the per-rollout table `pretty()` prints. A group whose scores are all equal carries no signal — the loop reports that honestly instead of pretending.

**Goal.** One real, judge-driven update you can see move the model (loss, adv, logp before → after).

**Need.** The student loaded in section 6, the teacher, ~3-6 minutes for the rollouts on a T4.

In [ ]:
from experiments.selfimprove import grpo, tasks

task = tasks.catalog()[0]
root = tasks.scaffold_root()

provider_local.TEMPERATURE = 1.0
group, rewards = [], []
for i in range(4):
    wd = tasks.materialize(task, root)
    h = Harness(wd, model=provider_local.DEFAULT_MODEL, max_turns=4,
                budget_tokens=4_000_000, persist=False, enable_subagents=False)
    h.run(task["task"])
    system = h.system + traj.tool_block([t.spec for t in h.tools.values()])
    group.append({"messages": h.messages, "system": system})
    rewards.append(teacher.grade(h.messages))
    print(f"rollout {i}: score={rewards[-1]} turns={len(h.messages)}")

# LoRA head is always attached: section 10's SFT reuses this same head, and
# attaching it twice would double-wrap the model.
lora = grpo.add_lora(model)

if grpo.degenerate(rewards):
    print("no signal in this group - rewards", rewards,
          "- GRPO update skipped (LoRA still attached for section 10)")
else:
    step = grpo.grpo_step(lora, tok, group, rewards, system=None, device=device)
    print(grpo.pretty(step))


## 10. Offline learning: SFT on pooled goldens

**Logic.** The imitation path: drain the pool, turn each `{issue, golden}` into a two-turn user/assistant example, and run teacher-forced cross-entropy over assistant tokens only — same masking `grpo.build_masked` enforces for GRPO, so both paths share one faithfulness guarantee.

**Goal.** A loss that visibly decreases from before to after, and a pool that is now empty and ready for new sessions.

**Need.** At least one banked golden from section 8, and the trained `lora` head from section 9 (attaching LoRA twice would double-wrap the model).

In [ ]:
from experiments.selfimprove import offline

goldens = pool.drain()
if not goldens:
    print("pool empty — close a chat session first")
else:
    result = offline.sft_step(lora, tok, goldens, system=q.system, device=device)
    print(f"{result['examples']} goldens: CE {result['loss_before']} -> "
          f"{result['loss_after']}")

## 11. The UI: one screen for all three modes

**Logic.** The last thing is the control surface. `colab_gradio.build_app` wires the same student, teacher and pool to three tabs: chat with the student (banking its golden on close), offline learning (drain pool + SFT), and online learning (a toggleable background loop that rolls out one task at a time, judges, takes one GRPO step, stops between tasks). The status line shows model, LoRA state, pool count and loop state.

**Goal.** A shareable Gradio link to drive the whole experiment interactively.

**Need.** Everything above; gradio from section 1; running this cell blocks while the UI is up.

In [ ]:
from experiments.selfimprove import colab_gradio

app, demo = colab_gradio.build_app(
    workdir="/content/irina_app", pool_path=POOL_PATH)
print("teacher binding:", teacher.binding())
demo.launch(share=True, debug=True)


## What you just built

- **Acting:** Qwen2.5-Coder-0.5B taking over the harness as the student — same loop, tools and file sandbox as DeepSeek had.
- **Teaching:** DeepSeek judging transcripts (GRPO reward) and distilling goldens (SFT labels), never seeing hidden tests.
- **Learning:** one GRPO step online, one SFT pass offline, both masked to assistant turns only by the same append-only chat conversion.
- **Persistence:** every banked golden sits in `insight_pool.json` on your Drive, surviving runtime resets.

The next iteration is what the Gradio app does: chat, pool accumulates, the offline button fine-tunes, the online loop pushes the reward curve. Each successful learning event is a commit your future harvest can read back.